In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
from IPython.display import display

In [ ]:
REQUIRED_FILES = (
    "Afluencia Diaria del Metro (Desglosada).csv",
    "Afluencia Diaria del Metrobús (Desglosada).csv",
    "Afluencia Preliminar en Transporte Publico (histórico).csv",
    "Diccionario Afluencia Desglosada - MB.csv",
    "Diccionario de datos (Afluencia desglosada).csv",
    "Diccionario de datos ingresos STCM.xlsx",
    "Ingresos del Sistema de Transporte Colectivo Metro.csv",
    "Viajes diarios del sistema Ecobici.csv",
    "Viajes Ecobici 2026-01.csv",
    "Viajes mensuales desglosados del sistema Ecobici.csv",
)

def localizar_datos() -> Path:
    """Devuelve la carpeta datasets/ de Movilidad-Urbana-del-Sistema-de-Transporte-CDMX o indica cómo prepararla."""
    candidatos = []
    definida = os.environ.get("MOVILIDAD_DATA_DIR")
    if definida:
        candidatos.append(Path(definida).expanduser())

    ubicacion = Path.cwd().resolve()
    for base in (ubicacion, *ubicacion.parents):
        candidatos.extend([
            base / "Movilidad-Urbana-del-Sistema-de-Transporte-CDMX" / "Datasets",
            base / "Datasets",
            base.parent / "Movilidad-Urbana-del-Sistema-de-Transporte-CDMX" / "Datasets",
        ])

    for ruta in candidatos:
        if ruta.is_dir() and all((ruta / nombre).is_file() for nombre in REQUIRED_FILES):
            return ruta
    raise FileNotFoundError(
        "No encontré los CSV de Movilidad-Urbana. Ejecuta 'git clone "
        "https://github.com/Juan-Gallegos7/Movilidad-Urbana-del-Sistema-de-Transporte-CDMX' en la "
        "carpeta del curso y coloca este notebook en una carpeta hermana "
        "del repositorio; o establece MOVILIDAD_DATA_DIR con la ruta a datasets/."
    )


DATA_DIR = localizar_datos()
print("Datos encontrados en:", DATA_DIR)

In [ ]:
ruta = Path("C:/Users/ricar/iCloudDrive/Diplomado_Modelos_Est_y_Mat/Modulo_1/Proyecto/Movilidad-Urbana-del-Sistema-de-Transporte-CDMX/Datasets")
for nombre in REQUIRED_FILES:
    existe = ( ruta / nombre).is_file()
    print(nombre, "->", existe)

In [ ]:

import chardet

with open(
    DATA_DIR / "Afluencia Diaria del Metro (Desglosada).csv",
    "rb"
) as f:
    resultado = chardet.detect(f.read(100000))

print(resultado)

In [ ]:
Afluencia_Metro = pd.read_csv(DATA_DIR / "Afluencia Diaria del Metro (Desglosada).csv")
Afluencia_Metrobus = pd.read_csv(DATA_DIR / "Afluencia Diaria del Metrobús (Desglosada).csv")
Afluencia_TP_historico = pd.read_csv(DATA_DIR / "Afluencia Preliminar en Transporte Publico (histórico).csv")
Ingresos_SCTM = pd.read_csv(DATA_DIR / "Ingresos del Sistema de Transporte Colectivo Metro.csv")
#Dic_Afluencia = pd.read_csv(DATA_DIR / "Diccionario Afluencia Desglosada - MB.csv", encoding="ISO-8859-1")
#Dic_datos_Afluencia = pd.read_csv(DATA_DIR / "Diccionario de datos (Afluencia desglosada).csv", encoding="ISO-8859-1")
Dic_ingresos_SCTM = pd.read_excel(DATA_DIR / "Diccionario de datos ingresos STCM.xlsx")
#Viajes_Ecobici_202601 = pd.read_csv(DATA_DIR / "Viajes Ecobico 2026-01.csv")
#Viajes_Ecobici_mensual = pd.read_csv(DATA_DIR / "Viajes mensuales desglosados del sistema Ecobici.csv")

for nombre, tabla in {
    "Afluencia_Metro": Afluencia_Metro,
    "Afluencia_Metrobus": Afluencia_Metrobus,
    "Afluencia_TP_historico": Afluencia_TP_historico,
    #"Dic_Afluencia": Dic_Afluencia,
    #"Dic_datos_Afluencia": Dic_datos_Afluencia,
    "Dic_ingresos_SCTM": Dic_ingresos_SCTM,
    "Ingresos_SCTM": Ingresos_SCTM,
    #"Viajes_Ecobici_202601": Viajes_Ecobici_202601,
    #"Viajes_Ecobici_mensual": Viajes_Ecobici_mensual,
}.items():
    print(f"{nombre:12s}: {len(tabla):>8,} filas × {tabla.shape[1]:>2} columnas")

display(Afluencia_Metro.head(3))
display(Afluencia_Metrobus.head(3))
display(Afluencia_TP_historico.head(3))
#display(Dic_Afluencia.head())
#display(Dic_datos_Afluencia.head())
display(Dic_ingresos_SCTM.head())
display(Ingresos_SCTM.head(3))
#display(Viajes_Ecobici_202601.head(3))
#display(Viajes_Ecobici_mensual.head(3))

In [ ]:
display(Afluencia_Metro.isna().sum().sort_values(ascending=False).head(8).to_frame("Faltantes Metro"))
display(Afluencia_Metrobus.isna().sum().sort_values(ascending=False).head(8).to_frame("Faltantes Metrobus"))
display(Afluencia_TP_historico.isna().sum().sort_values(ascending=False).head(8).to_frame("Faltantes Historico"))

Dado que los valores de la variable linea están incorrectos en formato, es necesario corregir la etiqueta "LÃ­nea" por "Linea", para mantener consistencia.

In [ ]:
Afluencia_Metro["linea"]=Afluencia_Metro["linea"].str.replace("LÃ­nea","Linea", regex=False)

In [ ]:
print("Registros por línea (Metro): ", Afluencia_Metro.groupby("linea").size())
print("Registros por mes (Metro): ", Afluencia_Metro.groupby("mes").size())
print("Registros por mes (Año): ", Afluencia_Metro.groupby("anio").size())
print("Registros por mes (Estacion): ", Afluencia_Metro.groupby("estacion").size())
print("Registros por mes (tipo_pago): ", Afluencia_Metro.groupby("tipo_pago").size())

Dado que los valores de la variable linea están incorrectos en formato, es necesario corregir la etiqueta "Línea" por "Linea", para mantener consistencia.

In [ ]:
Afluencia_Metrobus["linea"]=Afluencia_Metrobus["linea"].str.replace("Línea","Linea", regex=False)

In [ ]:
print("Registros por línea (Metrobus): ", Afluencia_Metrobus.groupby("linea").size())
print("Registros por mes (Metrobus): ", Afluencia_Metrobus.groupby("mes").size())
print("Registros por Año (Metrobus): ", Afluencia_Metrobus.groupby("anio").size())
print("Registros por Tipo pago (Metrobus): ", Afluencia_Metrobus.groupby("tipo_pago").size())

In [ ]:
print("Registros por organismo (Metro): ", Afluencia_TP_historico.groupby("organismo").size())
print("Registros por servicio (Metro): ", Afluencia_TP_historico.groupby("linea_servicio").size())
print("Registros por dia (Año): ", Afluencia_TP_historico.groupby("dia").size())